# **LLM Meal Planner**

In [1]:
import os
import sys
from datetime import datetime, timedelta

from openai import OpenAI

sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv

from engines import LLMMealPlanner, make_preferences, make_recipe
from models import (
    MealPlanningEnvironment,
    NutritionalInformation,
    Pantry,
    PantryIngredient,
)
from models.DietaryTag import DietaryTag

load_dotenv()

True

In [2]:
preferences = make_preferences()

In [3]:
recipes = [
    make_recipe(
        "Grilled Chicken & Rice",
        {"chicken breast": 200, "rice": 150, "olive oil": 10},
        calories=620,
        protein=52,
    ),
    make_recipe(
        "Vegetable Stir Fry",
        {"broccoli": 200, "carrot": 100, "soy sauce": 20, "rice": 150},
        calories=380,
        protein=12,
        tags=[DietaryTag.VEGAN, DietaryTag.VEGETARIAN],
    ),
    make_recipe(
        "Pasta Bolognese",
        {"pasta": 200, "ground beef": 150, "tomato": 100, "onions": 50},
        calories=710,
        protein=38,
    ),
    make_recipe(
        "Scrambled Eggs on Toast",
        {"eggs": 150, "butter": 15, "bread": 80},
        calories=420,
        protein=22,
        tags=[DietaryTag.VEGETARIAN],
    ),
    make_recipe(
        "Lentil Soup",
        {"dried brown lentils": 150, "carrot": 80, "celery": 60, "garlic": 10},
        calories=310,
        protein=20,
        tags=[DietaryTag.VEGAN, DietaryTag.VEGETARIAN, DietaryTag.GLUTEN_FREE, DietaryTag.LACTOSE_FREE],
    ),
]

print(f"{len(recipes)} recipes loaded:")
for i, r in enumerate(recipes):
    print(f"  {i}: {r.name}  ({r.nutritional_information.calories} kcal, {r.nutritional_information.protein}g protein)")


5 recipes loaded:
  0: Grilled Chicken & Rice  (620 kcal, 52g protein)
  1: Vegetable Stir Fry  (380 kcal, 12g protein)
  2: Pasta Bolognese  (710 kcal, 38g protein)
  3: Scrambled Eggs on Toast  (420 kcal, 22g protein)
  4: Lentil Soup  (310 kcal, 20g protein)


In [4]:
CURRENT_DATE = datetime.now()

pantry = Pantry()

pantry_items = [
    ("chicken breast", 800, 2.50, 3),  # expiring soon
    ("rice", 1000, 0.80, 14),
    ("broccoli", 400, 1.20, 5),
    ("eggs", 300, 0.30, 7),
]

for name, quantity, cost, days in pantry_items:
    pi = PantryIngredient(
        name=name,
        nutritional_information=NutritionalInformation(),
        estimated_expiration_date=CURRENT_DATE + timedelta(days=days),
        estimated_financial_cost=cost,
    )
    pantry.add(pi, quantity)

pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-30
	Estimated Financial Cost per 100g: EUR 2.50
	Nutritional Information:
		Calories: None kcal
		Carbohydrates: None g
		Sugar: None g
		Protein: None g
		Fat: None g
		Saturated Fat: None g
		Fiber: None g
		Sodium: None mg
		Gluten Free: No
		Lactose Free: No
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-11
	Estimated Financial Cost per 100g: EUR 0.80
	Nutritional Information:
		Calories: None kcal
		Carbohydrates: None g
		Sugar: None g
		Protein: None g
		Fat: None g
		Saturated Fat: None g
		Fiber: None g
		Sodium: None mg
		Gluten Free: No
		Lactose Free: No
		Vegetarian: No
		Vegan: No
---
---
Quantity: 400 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-02
	Estimated Financial Cost per 100g: EUR 1.20
	Nutritional Information:
		Calories: None kcal
		Carbohydrates: None g
		Sugar: None g
		Protein: None g
		Fat: None g
		Satu

In [5]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=recipes,
    pantry=pantry,
    preferences=preferences,
)

print(f"{len(meal_planning_environment.recipes)} recipes available after dietary filtering")

5 recipes available after dietary filtering


In [6]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

planner = LLMMealPlanner(meal_planning_environment, llm_client=client, prompt_filepath="./LLMMealPlannerPrompt.txt")

In [7]:
print(planner.prompt)

You are a meal planning assistant. Your task is to select a 7-day meal plan (3 meals per day = 21 meals total) from a provided list of recipes, optimised according to the user's preferences and pantry.

You will be given:
1. A numbered list of available recipes (index, name, dietary tags, calories, protein, key ingredients).
2. The user's pantry: ingredients currently in stock with their available quantities (in grams), with days until expiry and estimated financial cost.
3. The user's preferences: daily calorie target, daily protein target, weekly grocery budget, and any dietary restrictions (vegetarian, vegan, gluten-free, lactose-free).

This list of input should always remain untouched. You must not add any new recipes or ingredients, and you must not modify the pantry or user preferences. Your task is solely to select the optimal combination of recipes from the provided list based on the given information.

---

YOUR GOAL

Select exactly 21 recipe indices (integers) from the provi

In [8]:
best_meal_plan, best_fitness_score = planner.generate_meal_plan()

: 

: 

In [ ]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Grilled Chicken & Rice,Pasta Bolognese,Lentil Soup
1,2,Grilled Chicken & Rice,Lentil Soup,Pasta Bolognese
2,3,Vegetable Stir Fry,Pasta Bolognese,Lentil Soup
3,4,Vegetable Stir Fry,Pasta Bolognese,Lentil Soup
4,5,Scrambled Eggs on Toast,Pasta Bolognese,Lentil Soup
5,6,Scrambled Eggs on Toast,Lentil Soup,Pasta Bolognese
6,7,Pasta Bolognese,Lentil Soup,Lentil Soup


In [ ]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,400,400,2d
1,rice,1000,600,400,13d
2,broccoli,400,400,0,4d
3,eggs,300,300,0,6d


In [ ]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 14
Total cost: €69.20


,Ingredient,Quantity to Buy (g),Cost (€)
0,bread,160.0,1.6
1,broccoli,400.0,4.8
2,butter,30.0,0.3
3,carrot,840.0,8.4
4,celery,480.0,4.8
5,dried brown lentils,1200.0,12.0
6,eggs,300.0,0.9
7,garlic,80.0,0.8
8,ground beef,1050.0,10.5
9,olive oil,20.0,0.2


In [ ]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,1640 kcal,110 g,-360.0 kcal,60.0 g
Day 2,1640 kcal,110 g,-360.0 kcal,60.0 g
Day 3,1400 kcal,70 g,-600.0 kcal,20.0 g
Day 4,1400 kcal,70 g,-600.0 kcal,20.0 g
Day 5,1440 kcal,80 g,-560.0 kcal,30.0 g
Day 6,1440 kcal,80 g,-560.0 kcal,30.0 g
Day 7,1330 kcal,78 g,-670.0 kcal,28.0 g
